<a href="https://colab.research.google.com/github/22417010/Gemini-Intelligence-Benchmarking/blob/main/Gemini%20Intelligence%20Benchmarking%20%E2%94%82%E2%94%82%20%5BRAG%20vs.%20Long%20Context%20System%5D.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

╭─────────────────────────────────────╮
│ 🧪 Gemini Intelligence Benchmarking │
│ [RAG vs. Long Context System]       │
╰─────────────────────────────────────╯

Saving peerj-cs-08-1001.pdf to peerj-cs-08-1001 (1).pdf


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Output()

✓ تم فهرسة المستند بنجاح (63 قطعة نصية).

أدخل سؤالك (أو 'exit' للخروج): 

A hybrid forecasting model using LSTM and Prophet for energy consumption with decomposition of time series data


Output()

NotFoundError: Error code: 404 - {'error': {'message': 'No endpoints found for google/gemini-flash-1.5-8b.', 'code': 404}, 'user_id': 'user_36AH7IFobalzLxHdoODkoatvCsB'}

# ============================================================
# Project: Gemini Intelligence Benchmarking (RAG vs. Long Context)
# Developer: Gemini Adaptive Collaborator
# Platform: Google Colab / OpenRouter API
# ============================================================



# 1. تثبيت المكتبات المطلوبة

In [ ]:

!pip install -q openai pypdf faiss-cpu sentence-transformers rich

import os
import time
import numpy as np
import faiss
from google.colab import files
from openai import OpenAI
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
from rich.console import Console
from rich.panel import Panel
from rich.table import Table

console = Console()

# 2. الإعدادات (Configuration)
# تم استخدام مفتاح OpenRouter الخاص بك لضمان العمل بدون دفع

In [7]:

OS_API_KEY = "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"
MODEL_RAG = "nvidia/nemotron-3-nano-omni-30b-a3b-reasoning:free"
MODEL_LONG = "google/gemini-2.0-flash-001"# تم التحديث من 1.5-8b إلى 2.0 المتاح عندك

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=OS_API_KEY,
)




# 3. محرك معالجة الوثائق (Document Processing Engine)

In [4]:

class GeminiRAGEngine:
    def __init__(self):
        # استخدام موديل محلي ومجاني لتحويل النصوص إلى متجهات
        self.embed_model = SentenceTransformer('all-MiniLM-L6-v2')
        self.index = None
        self.chunks = []
        self.full_text = ""

    def process_pdf(self, pdf_path):
        """قراءة الـ PDF وتقسيمه هندسياً"""
        reader = PdfReader(pdf_path)
        text = ""
        for page in reader.pages:
            text += page.extract_text() + "\n"
        self.full_text = text

        # تقسيم النص: Chunk size 800 with 100 overlap
        self.chunks = [text[i:i + 800] for i in range(0, len(text), 700)]

        # بناء فهرس FAISS للبحث السريع
        embeddings = self.embed_model.encode(self.chunks, normalize_embeddings=True)
        dimension = embeddings.shape[1]
        self.index = faiss.IndexFlatIP(dimension)
        self.index.add(embeddings.astype(np.float32))
        return len(self.chunks)

    def retrieve(self, query, k=3):
        """استرجاع أفضل القطع النصية المشابهة للسؤال"""
        query_emb = self.embed_model.encode([query], normalize_embeddings=True).astype(np.float32)
        _, indices = self.index.search(query_emb, k)
        return [self.chunks[i] for i in indices[0]]


# 4. وظيفة المقارنة المعيارية (Benchmarking Logic)


In [5]:
def run_benchmark(engine, query):
    # --- المسار الأول: RAG Approach ---
    start_rag = time.time()
    context_chunks = engine.retrieve(query)
    context_text = "\n---\n".join(context_chunks)

    rag_prompt = f"Answer based ONLY on the context:\n{context_text}\nQuestion: {query}"
    rag_res = client.chat.completions.create(
        model=MODEL_RAG,
        messages=[{"role": "user", "content": rag_prompt}]
    )
    time_rag = time.time() - start_rag

    # --- المسار الثاني: Long Context Approach ---
    start_long = time.time()
    long_prompt = f"Full Document Context:\n{engine.full_text}\nQuestion: {query}"
    long_res = client.chat.completions.create(
        model=MODEL_LONG,
        messages=[{"role": "user", "content": long_prompt}]
    )
    time_long = time.time() - start_long

    return {
        "rag": {"ans": rag_res.choices[0].message.content, "time": time_rag},
        "long": {"ans": long_res.choices[0].message.content, "time": time_long}
    }


# 5. واجهة التشغيل النهائية (Main Interface)


In [ ]:

def main():
    console.print(Panel.fit("🧪 Gemini Intelligence Benchmarking\n[RAG vs. Long Context System]", style="bold cyan"))

    # رفع الملف
    uploaded = files.upload()
    if not uploaded: return
    file_path = list(uploaded.keys())[0]

    # المعالجة
    engine = GeminiRAGEngine()
    with console.status("[bold yellow]جاري بناء الفهرس الدلالي..."):
        num_chunks = engine.process_pdf(file_path)
    console.print(f"[bold green]✓ تم فهرسة المستند بنجاح ({num_chunks} قطعة نصية).[/bold green]")

    while True:
        raw_input = console.input("\n[bold magenta]أدخل سؤالك (أو 'exit' للخروج): [/bold magenta]")
        query = str(raw_input) # حل مشكلة AttributeError

        if query.lower() in ['exit', 'خروج', 'quit']: break

        with console.status("[bold green]جاري تشغيل المقارنة المعيارية..."):
            results = run_benchmark(engine, query)

        # عرض النتائج في جدول احترافي
        table = Table(title="📊 المقارنة المعيارية للنتائج")
        table.add_column("الخاصية (Metric)", style="cyan")
        table.add_column("نهج الـ RAG (المحسن)", style="green")
        table.add_column("نهج السياق الطويل (الخام)", style="blue")

        table.add_row("زمن الاستجابة", f"{results['rag']['time']:.2f} ثانية", f"{results['long']['time']:.2f} ثانية")
        table.add_row("حجم السياق المرسل", "Top-K Segments", "Full Document Text")

        console.print(table)

        # عرض الإجابات في صناديق
        console.print(Panel(results['rag']['ans'], title="إجابة RAG (Nvidia Nemotron)", border_style="green"))
        console.print(Panel(results['long']['ans'], title="إجابة Long Context (Gemini Flash)", border_style="blue"))

# تشغيل النظام
if __name__ == "__main__":
    main()

╭─────────────────────────────────────╮
│ 🧪 Gemini Intelligence Benchmarking │
│ [RAG vs. Long Context System]       │
╰─────────────────────────────────────╯

Saving kong2019.pdf to kong2019.pdf


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Output()

...

Output()

                     📊 المقارنة المعيارية للنتائج                      
┏━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ الخاصية (Metric)  ┃ نهج الـ RAG (المحسن) ┃ نهج السياق الطويل (الخام) ┃
┡━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ زمن الاستجابة     │ 3.68 ثانية           │ 8.11 ثانية                │
│ حجم السياق المرسل │ Top-K Segments       │ Full Document Text        │
└───────────────────┴──────────────────────┴───────────────────────────┘

╭────────────────────────────────────────── إجابة RAG (Nvidia Nemotron) ──────────────────────────────────────────╮
│ The research methodology comprised thefollowing steps:                                                          │
│                                                                                                                 │
│ 1. **Data collection** – Daily historical electricity‑load records for individual households were gathered and  │
│ treated as labeled sequences.                                                                                   │
│ 2. **Clustering** – A density‑based spatial clustering technique was applied to these sequences to group        │
│ households that exhibit similar daily load patterns (i.e., similar “profiles”).                                 │
│ 3. **Modeling** – Each clustered group was modeled as a Markov chain, capturing the temporal dynamics of load   │
│ within that group.                                                                                              │
│ 4. **Prediction** – For day‑ahead forecasting, the day‑ahead label was sampled from the fitted Markov chain,    │
│ and the cluster mean load at the relevant time points was used as the prediction value.                         │
│                                                                                                                 │
│ Overall, the approach leverages clustering to reduce forecasting error by grouping households with similar      │
│ usage behaviors, then applies a Markov‑chain model to each group for day‑ahead load prediction.                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────── إجابة Long Context (Gemini Flash) ───────────────────────────────────────╮
│ Okay, here's a summary of the research methodology used in the document, broken down by section:                │
│                                                                                                                 │
│ **Overall Goal:** The research aims to improve short-term load forecasting for individual residential           │
│ households by using a Long Short-Term Memory (LSTM) recurrent neural network (RNN).                             │
│                                                                                                                 │
│ **I. Introduction**                                                                                             │
│ *   States the importance of short-term load forecasting, especially with increasing renewable energy           │
│ penetration.                                                                                                    │
│ *   Highlights the challenge of forecasting individual customer loads due to volatility.                        │
│ *   Introduce the usage of LSTM RNN to tackle this.                                                             │
│                                                                                                                 │
│ **II. Literature Study**                                                                                        │
│ *   Reviews existing load forecasting methods focusing system-level forecasting methodologies.                  │
│ *   Notes the gap in research on individual customer load forecasting.                                          │
│ *   Discusses existing work such as deep learning techniques being employed.                                    │
│                                                                                                                 │
│ **III. Exploratory Data Analysis and Problem Identification**                                                   │
│ *   **Dataset:** Uses a subset of the Smart Grid Smart City (SGSC) project data, focusing on 69 households with │
│ hot water systems.                                                                                              │
│ *   **Data Analysis:**                                                                                          │
│     *   Compares load patterns at the system level vs. individual household level.                              │
│     *   Uses **DBSCAN (Density-Based Spatial Clustering of Applications with Noise)**, a density-based          │
│ clustering technique, to analyze the consistency of daily load profiles, each represented as a 48-dimensional   │
│ sample (48 half-hourly readings).                                                                               │
│     *   Applies DBSCAN to the aggregated load and each of the 69 households to cluster the daily power          │
│ profiles. The parameters for DBSCAN were: Euclidean distance, eps=10% of daily energy consumption, and          │
│ MinPts=2.                                                                                                       │
│     *   Analyzes the number of outliers (days that don't fit into any cluster) as a measure of inconsistency.   │
│     *   Justifies the use of LSTM due to its capacity to learn temporal correlations                            │
│                                                                                                                 │
│ **IV. The Forecasting Framework Based on LSTM**                                                                 │
│ *   **LSTM Model:**                                                                                             │
│     *   Explains the structure of LSTM.                                                                         │
│     *   Describes how LSTM learns previous observation

أدخل سؤالك (أو 'exit' للخروج): 